# Task 1 - build the dataset

Fill in the path to your local nuScenes v1.0-mini in the next cell, then **Run All**.

At the end you get one merged YOLO dataset - `0 cone, 1 barrier, 2 stop_sign` - with
`configs/data.yaml` written and ready for Task 2.

| Step | What it does |
| --- | --- |
| 1 | nuScenes -> YOLO labels, plus a `meta/` sidecar per image holding `K`, camera height, pitch and ground-truth depth (Task 3 needs these) |
| 2 | downloads only the ~2,000 COCO 2017 images that contain a stop sign, as class 2 |
| 3 | pseudo-labels stop signs on the nuScenes frames, so the merge does not teach the model they are background |
| 4 | validates the pairing and counts, writes `configs/data.yaml` |
| 5 | sanity checks: per-class and per-scene counts, per-camera `K`, positive depth, 20 drawn images |

Re-running is safe. Step 1 clears what it wrote last time, step 2 skips images already
downloaded, and step 3 strips its own previous labels before adding new ones.

In [ ]:
# ============================ CONFIG - edit this cell ============================

# Your local nuScenes v1.0-mini. Can be the dataroot itself or any folder above
# it: the v1.0-mini/ + samples/ pair is found inside.
NUSCENES = r"C:\path\to\nuscenes-mini"

# Scenes held out as val. The devkit's own mini_val (scene-0103, scene-0916)
# contains no barriers at all, so barrier AP would be unmeasurable. This pair
# puts 218 cones and 584 barriers in val and keeps the richest scene (0061) for
# training. Step 5 prints per-scene counts if you want to choose differently.
VAL_SCENES = ["scene-0655", "scene-0553"]

PSEUDO_LABEL = True      # step 3; needs ultralytics. Slow on CPU.
DEVICE = ""              # "" = auto (CUDA if present), or "cpu", or "0"
LINK_MODE = "auto"       # how images reach images/<split>/: hardlink, else symlink, else copy
BALANCED_LIST = False    # also write a class-balanced training list (optional)

# ================================================================================

In [ ]:
import os, sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO))

from common.env import find_nuscenes_root, writable_root

NUSCENES = find_nuscenes_root(NUSCENES)   # raises here if the path is wrong
OUT = writable_root() / "datasets/yolo_3class"
VAL_ARG = " ".join(VAL_SCENES)
# an empty --device would reach argparse as a literal "" on some shells
DEVICE_ARG = f'--device {DEVICE}' if DEVICE else ''

print("repo     :", REPO)
print("nuScenes :", NUSCENES)
print("output   :", OUT)
print("val      :", VAL_ARG)

In [ ]:
# 1. nuScenes -> YOLO. Split is by scene; a frame split would leak, since
#    keyframes 0.5 s apart show the same cones.
!python -m data.nuscenes_to_yolo --nuscenes-root "{NUSCENES}" --out "{OUT}" --val-scenes {VAL_ARG} --link-mode {LINK_MODE}

In [ ]:
# 2. COCO stop signs -> class 2, into the same tree. First run downloads the
#    annotation zip (~241 MB) and the stop-sign images (~350 MB); later runs skip both.
!python -m data.coco_stopsign --out "{OUT}" --link-mode {LINK_MODE}

In [ ]:
# 3. Pseudo-label the stop signs nuScenes never annotates.
if PSEUDO_LABEL:
    !python -m data.pseudo_label --out "{OUT}" --weights yolo11n.pt --conf 0.5 --imgsz 960 --batch 16 {DEVICE_ARG}
else:
    print("skipped - stop signs will come from COCO only, and nuScenes frames")
    print("containing one will be treated as background, which costs stop sign AP")

In [ ]:
# Spot-check the pseudo-labels - they are only as good as this check.
from IPython.display import Image, display

sheet = REPO / "results/pseudo_label_review.jpg"
if sheet.exists():
    display(Image(str(sheet)))
else:
    print("no stop signs were found on the nuScenes frames")

In [ ]:
# 4. Validate and write configs/data.yaml (absolute paths, so it is per machine).
!python -m data.merge --out "{OUT}"
print()
print((REPO / "configs/data.yaml").read_text())

In [ ]:
# 5. Sanity checks. Read the per-scene table: if a class shows 0 in val, change
#    VAL_SCENES above and re-run from step 1.
!python -m data.sanity_check --out "{OUT}" --draw 20 --dest results/sanity

In [ ]:
# Optional: repeat images so each class is seen a similar number of times per
# epoch, and write configs/data_balanced.yaml. Train with it via --run B1bal.
if BALANCED_LIST:
    !python -m data.balance --out "{OUT}" --threshold 0.5
else:
    print("skipped - training uses the natural class distribution")

In [ ]:
# Eyeball the boxes. Labels show the ground-truth depth where nuScenes has one.
import itertools

for p in itertools.islice(sorted((REPO / "results/sanity/val").glob("*.jpg")), 6):
    display(Image(str(p), width=900))

## Done when

- `val:` reports non-zero **cone, barrier and stop_sign**;
- the drawn boxes sit on the objects - a normalisation bug shows up here immediately;
- the contact sheet really contains stop signs;
- the sanity check ends with *all sanity checks passed*.

Then Task 2:

```
python -m eval.det_eval --run B0          # zero-shot floor
python -m detection.train --run B1        # first real training run
```